# Colab End-to-End: Hindi Punctuation, Ordering, Grouping & Translation

This notebook runs the project pipeline in Google Colab.

**Constraints:** No GPT/Gemini/Claude. Keep GPU memory < 1.5 GB if using GPU (CPU is recommended for compliance in this pipeline).

**IndicTrans2:** requires accepting model access on Hugging Face and providing `HF_TOKEN` in Colab secrets. If unavailable, translation falls back to MarianMT (`Helsinki-NLP/opus-mt-hi-en`).

## 1) Runtime setup (recommended: CPU for <1.5GB GPU constraint)
In Colab: `Runtime > Change runtime type` → select **CPU** (or GPU only if you will measure and stay under 1.5 GB VRAM).

In [1]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/gitanshu-chavda/Punctuation_Adder-and-Sentence-Ordering.git"
BRANCH = "sen-ord-1"
WORKDIR = pathlib.Path('/content/Punctuation_Adder-and-Sentence-Ordering')

if not WORKDIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(WORKDIR)])
else:
    subprocess.check_call(['git', '-C', str(WORKDIR), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(WORKDIR), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(WORKDIR), 'pull', '--ff-only', 'origin', BRANCH])

os.chdir(WORKDIR)
print('CWD:', os.getcwd())
print('Branch:', subprocess.check_output(['git','branch','--show-current'], text=True).strip())

CWD: /content/Punctuation_Adder-and-Sentence-Ordering
Branch: sen-ord-1


In [2]:
# Install dependencies (Colab/Linux)
!bash scripts/colab_setup.sh

[colab_setup] Installing Python dependencies for Colab...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 10.6 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.2.2
    Uninstalling pip-26.2.2:
      Successfully uninstalled pip-26.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 8.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 83.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [IndicTransToolkit]
[colab_setup] Done. Set HF_TOKEN for gated IndicTrans2 models if needed.


## 2) Configure Hugging Face token (for IndicTrans2 gated models)
1. Accept model access/license on Hugging Face for your chosen IndicTrans2 model (default: `ai4bharat/indictrans2-indic-en-dist-200M`).
2. In Colab, add a secret named `HF_TOKEN` (🔑 icon in left sidebar) and enable notebook access.
3. Run the cell below to load it into the environment.

In [3]:
import os

try:
    from google.colab import userdata  # type: ignore
    hf = userdata.get('HF_TOKEN')
    if hf:
        os.environ['HF_TOKEN'] = hf
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf
        print('HF_TOKEN loaded from Colab secrets.')
    else:
        print('HF_TOKEN secret not found/empty. IndicTrans2 may fall back to MarianMT if gated.')
except Exception as e:
    print('Colab userdata not available:', e)
    print('Set HF_TOKEN manually if needed: os.environ["HF_TOKEN"] = "..."')

# Translation backend selection: auto | indictrans2 | marian
os.environ.setdefault('TRANSLATION_BACKEND', 'auto')
os.environ.setdefault('INDICTRANS2_MODEL_ID', 'ai4bharat/indictrans2-indic-en-dist-200M')
print('TRANSLATION_BACKEND =', os.environ.get('TRANSLATION_BACKEND'))
print('INDICTRANS2_MODEL_ID =', os.environ.get('INDICTRANS2_MODEL_ID'))

Colab userdata not available: Secret HF_TOKEN does not exist.
Set HF_TOKEN manually if needed: os.environ["HF_TOKEN"] = "..."
TRANSLATION_BACKEND = auto
INDICTRANS2_MODEL_ID = ai4bharat/indictrans2-indic-en-dist-200M


## 3) Quick checks

In [4]:
!python setup_check.py

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
OK   data/raw/samples
OK   data/model_input
OK   data/punctuated
OK   data/ordered
OK   outputs
OK   reports
MISS models/punctuation/onnx/model_quantized.onnx

Status: MISSING ITEMS


## 4) Run pipeline stages

- **Punctuation**: ONNX CPU (`src/punctuation.py`)
- **Ordering**: mBERT pairwise scoring + Borda (`src/order_sentences.py`) — can be slow on CPU
- **Grouping + Translation**: 5-sentence groups (`src/group_and_translate.py`), IndicTrans2-preferred if available
- **Aligned outputs**: friend-style 4 files/sample (`scripts/generate_aligned_outputs.py`)

You can skip stages if artifacts already exist.

In [5]:
# Optional: regenerate punctuation (CPU)
!python src/punctuation.py

Traceback (most recent call last):
  File "/content/Punctuation_Adder-and-Sentence-Ordering/src/punctuation.py", line 288, in <module>
    restorer = PunctuationRestorer()
  File "/content/Punctuation_Adder-and-Sentence-Ordering/src/punctuation.py", line 48, in __init__
    raise FileNotFoundError(MODEL_FILE)
FileNotFoundError: models/punctuation/onnx/model_quantized.onnx


In [ ]:
# Ordering (mBERT pairwise+Borda) - may take a while on CPU
!python src/order_sentences.py

Loading multilingual BERT for pairwise ordering: bert-base-multilingual-cased
config.json: 100% 625/625 [00:00<00:00, 2.23MB/s]
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 201kB/s]
vocab.txt: 100% 996k/996k [00:00<00:00, 18.8MB/s]
tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 39.6MB/s]

model.safetensors: downloading bytes:  57% 409M/714M [00:02<00:02, 141MB/s, 34.7MB/s  ]
model.safetensors: downloading bytes:  90% 646M/714M [00:05<00:01, 50.1MB/s, 46.5MB/s  ]
model.safetensors: downloading bytes:  94% 672M/714M [00:05<00:00, 49.8MB/s, 46.6MB/s  ]
model.safetensors: reconstructing file:  66% 469M/714M [00:05<00:03, 77.0MB/s, 31.5MB/s  ]
model.safetensors: downloading bytes: 100% 676M/676M [00:07<00:00, 86.1MB/s, 46.5MB/s  ]
model.safetensors: reconstructing file: 100% 714M/714M [00:07<00:00, 90.9MB/s, 51.4MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 3886.25it/s]
[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                  

In [ ]:
# Grouping + translation (IndicTrans2 if configured/available, else MarianMT fallback)
!python src/group_and_translate.py

In [ ]:
# Generate aligned outputs (outputs/sample-N.sentences_hi/en, groups_hi/en)
!python scripts/generate_aligned_outputs.py

## 5) Metrics, integrity, and reports

In [ ]:
!python scripts/verify_integrity.py
!python scripts/evaluate_punctuation.py
!python scripts/compute_metrics.py
!python scripts/profile_resources.py --steps prepare,integrity,metrics

In [ ]:
import json, pathlib
for p in ['reports/evaluation.json','reports/run_summary.json','reports/resource_profile.json','data/final/translation_summary.json']:
    path = pathlib.Path(p)
    print('\n' + '='*80)
    print(p)
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        if p.endswith('evaluation.json'):
            print('Ordering metrics:', data.get('ordering', {}))
            print('Translation:', data.get('translation', {}))
        elif p.endswith('translation_summary.json'):
            print({k:data.get(k) for k in ['translation_backend','translation_model','group_size']})
        else:
            print(json.dumps({k:data.get(k) for k in list(data.keys())[:8]}, indent=2, ensure_ascii=False))
    else:
        print('Not found')

## 6) Inspect outputs

In [ ]:
from pathlib import Path
sample = 'sample-1'
for name in [f'outputs/{sample}.sentences_hi.txt', f'outputs/{sample}.sentences_en.txt', f'outputs/{sample}.groups_hi.txt', f'outputs/{sample}.groups_en.txt']:
    p = Path(name)
    print('\n' + '='*80)
    print(name, 'exists=', p.exists())
    if p.exists():
        lines = p.read_text(encoding='utf-8').splitlines()
        for ln in lines[:6]:
            print(ln[:200])

## 7) Download results (optional)

In [ ]:
# Zip key outputs/reports for download
!zip -r results_bundle.zip outputs reports data/final data/ordered data/punctuated README.md

try:
    from google.colab import files  # type: ignore
    files.download('results_bundle.zip')
except Exception as e:
    print('Download manually from file browser:', e)

## Notes
- **NULL markers (67 total)** are preserved; review `scripts/null_diagnostic.py` and `reports/null_punct_analysis.json`.
- **Ordering/translation quality** must be manually reviewed; metrics in reports are partial/proxy where noted.
- If IndicTrans2 is used, record model ID, environment, and any token/access steps for reproducibility (do not share tokens).
- For GPU runs, measure peak VRAM and ensure it stays < 1.5 GB per assignment constraint.